# Healthcare Data Analysis — Medallion Architecture

End-to-end ETL pipeline reading `healthcare_dataset.csv` from ADLS Gen2 and
processing it through **Bronze → Silver → Gold** layers in the `azuredatabricks1` catalog.

| Layer | Schema | Purpose |
|-------|--------|---------|
| **Bronze** | `azuredatabricks1.bronze` | Raw ingestion — data as-is from source |
| **Silver** | `azuredatabricks1.silver` | Cleaned, standardized, enriched |
| **Gold** | `azuredatabricks1.gold` | Aggregated, business-ready analytics |

> **Module**: All pipeline logic lives in `healthcare_medallion_pipeline.py`. This notebook
> serves as the interactive execution layer.

In [0]:
import sys
import importlib

MODULE_DIR = "/Workspace/Users/pavanreddy_adf@outlook.com/Azure_Databricks"
if MODULE_DIR not in sys.path:
    sys.path.insert(0, MODULE_DIR)

# Clear cached modules for clean re-import
for mod_name in ["ADLS_Databricks_Connection", "healthcare_medallion_pipeline"]:
    if mod_name in sys.modules:
        del sys.modules[mod_name]

# Clear any failed import caches
importlib.invalidate_caches()

# --- Import ADLS connectivity ---
import ADLS_Databricks_Connection as adls_module
adls_module.dbutils = dbutils
adls_module.spark = spark
adls_module.display = display
from ADLS_Databricks_Connection import ADLSManager

# --- Import Medallion Pipeline ---
import healthcare_medallion_pipeline as hmp_module
hmp_module.display = display
from healthcare_medallion_pipeline import HealthcareMedallionPipeline, PipelineConfig

print("✔ Imported ADLSManager, HealthcareMedallionPipeline, PipelineConfig")

In [0]:
# Configure ADLS connection
CONFIG_PATH = "/Workspace/Users/pavanreddy_adf@outlook.com/Azure_Databricks/etl_config.json"
adls = ADLSManager(CONFIG_PATH)
adls.configure_oauth()

# Initialize medallion pipeline
config = PipelineConfig()  # uses defaults; override if needed
pipeline = HealthcareMedallionPipeline(
    adls_manager=adls,
    spark=spark,
    display_fn=display,
    config=config,
)

print("✔ ADLS connection ready")
print("✔ Pipeline initialized with config:")
print(f"  Catalog : {config.catalog}")
print(f"  Source  : {config.source_file_path}")

In [0]:
pipeline.setup_catalog_and_schemas()
print("✔ Medallion architecture schemas ready")

## 🟥 Bronze Layer — Raw Ingestion
Ingest raw CSV from ADLS Gen2 into `azuredatabricks1.bronze.healthcare_raw` with no transformations.

In [0]:
df_bronze = pipeline.ingest_bronze()
print(f"\n✔ Bronze layer complete  |  {pipeline._row_counts[config.bronze_fqn]:,} rows")
display(df_bronze.limit(10))

In [0]:
df_bronze_verify = spark.table(config.bronze_fqn)
print(f"Bronze table row count: {df_bronze_verify.count():,}")
df_bronze_verify.printSchema()
display(df_bronze_verify.limit(10))

## 🟨 Silver Layer — Cleansed and Enriched
Transformations applied:
- Standardize `name` to proper case
- Filter out invalid billing amounts (negative values)
- Remove duplicate records (window-based deduplication)
- Add `length_of_stay` computed column (days between admission and discharge)
- Add `ingestion_timestamp` audit column

In [0]:
df_silver = pipeline.transform_silver()
print(f"\n✔ Silver layer complete  |  {pipeline._row_counts[config.silver_fqn]:,} rows")
df_silver.printSchema()
display(df_silver.limit(10))

In [0]:
# Verify row count
df_silver_verify = spark.table(config.silver_fqn)
print(f"Silver table row count: {df_silver_verify.count():,}")

# Null counts per column
print("\n--- Null Counts per Column ---")
null_counts = pipeline.verify_silver_quality()
display(null_counts)

# Sample records
print("\n--- Sample Silver Records ---")
display(df_silver_verify.limit(10))

## 🟩 Gold Layer — Business-Ready Aggregations
Three analytics tables built from the silver layer:
- **Condition Summary** — patient counts, avg billing, avg length of stay by medical condition
- **Admission Summary** — metrics grouped by admission type and insurance provider
- **Hospital Performance** — hospital-level KPIs

In [0]:
gold_tables = pipeline.build_gold_layer()
print(f"\n✔ Gold layer complete  |  {len(gold_tables)} tables written")

In [0]:
print("--- Medical Condition Summary ---")
display(spark.table(config.gold_condition_fqn))

In [0]:
print("--- Admission Type Summary ---")
display(spark.table(config.gold_admission_fqn))

In [0]:
print("--- Hospital Performance ---")
display(spark.table(config.gold_hospital_fqn))

In [0]:
pipeline.print_summary()

In [0]:
# Uncomment below to run the entire pipeline in a single call
# This replaces all individual cells above with one-shot execution
#
# summary = pipeline.run()
# print(summary)

print("✔ Use pipeline.run() to execute all layers in a single call")